<a href="https://colab.research.google.com/github/Tharun-C0/URBAN-SOUND-CLASSIFICATION-CNN-RANDOM-FOREST/blob/main/SOUND.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "numpy<2"


In [ ]:
!pip install resampy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.1 MB/s eta 0:00:00


In [ ]:
!pip install flask-cors

In [ ]:
!pip install pyngrok

In [ ]:
import os
import tarfile
import urllib.request
import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import joblib

# 1. SETUP ENVIRONMENT
# Packages are managed by uv locally

from flask import Flask, request, jsonify, render_template_string
from flask_cors import CORS
from pyngrok import ngrok
import threading

# Kaggle Credentials Here (Make sure no spaces in Username natively if Kaggle errors, but using provided):
os.environ['KAGGLE_USERNAME'] = "Mohamed Ansari"
os.environ['KAGGLE_KEY'] = "KGAT_85d6205a61ee7397f544f02cd3220365"

DATASET_DIR = "UrbanSound8K"

def train_model():
    print("--- 1. DOWNLOADING DATASET VIA KAGGLE ---")
    if not os.path.exists(DATASET_DIR) and not os.path.exists("UrbanSound8K.csv"):
        os.system("kaggle datasets download -d chrisfilo/urbansound8k")
        os.system("unzip -q urbansound8k.zip -d .")
        print("Extracted!")

    metadata_path = os.path.join(DATASET_DIR, "metadata", "UrbanSound8K.csv") if os.path.exists(os.path.join(DATASET_DIR, "metadata", "UrbanSound8K.csv")) else "UrbanSound8K.csv"
    audio_path = os.path.join(DATASET_DIR, "audio") if os.path.exists(os.path.join(DATASET_DIR, "audio")) else "."

    print("--- 2. EXTRACTING FEATURES (This takes a little while) ---")
    df = pd.read_csv(metadata_path)
    # Using a subset of 1500 for speed in web app demo, change to len(df) for full model
    subset_df = df.sample(n=min(1500, len(df)), random_state=42)

    X, y = [], []
    for index, row in subset_df.iterrows():
        file_path = os.path.join(audio_path, f'fold{row["fold"]}', row["slice_file_name"])
        try:
            audio, sample_rate = librosa.load(file_path)
            mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
            X.append(np.mean(mfccs, axis=1))
            y.append(row["class"])
        except Exception:
            pass

    print("--- 3. TRAINING RANDOM FOREST MODEL ---")
    X, y = np.array(X), np.array(y)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_scaled, y)

    joblib.dump(rf_model, 'app_model.pkl')
    joblib.dump(scaler, 'app_scaler.pkl')
    print("Model Trained and Saved Successfully!")

# Try to train the model first if it doesn't exist
if not os.path.exists('app_model.pkl'):
    train_model()

# Load Model Globally
print("Loading model for Web Service...")
model = joblib.load('app_model.pkl')
scaler = joblib.load('app_scaler.pkl')

# ==============================================================================
# FLASK WEB SERVER
# ==============================================================================
app = Flask(__name__)
CORS(app)

# The HTML application we built, injected directly so it serves universally!
HTML_CONTENT = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Sound Intelligence</title>

  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Space+Mono:ital,wght@0,400;0,700;1,400&family=Syne:wght@400..800&display=swap" rel="stylesheet">

  <style>
    :root {
      --bg: #080c18;
      --surface: rgba(255, 255, 255, 0.035);
      --border: rgba(120, 100, 255, 0.28);
      --accent1: #7b5fff; --accent2: #d060cc; --accent3: #6ec6f5;
      --safe: #2ecc71; --warning: #e67e22; --critical: #e74c3c;
      --text: #e8e4ff; --text-muted: #7b78a8;
      --font-body: 'Syne', sans-serif; --font-mono: 'Space Mono', monospace;
      --spring: cubic-bezier(0.34, 1.56, 0.64, 1);
      --gradient-main: linear-gradient(135deg, var(--accent3), var(--accent2), var(--accent1));
      --gradient-text: linear-gradient(90deg, var(--accent3), var(--accent2), var(--accent1));
    }
    * { margin: 0; padding: 0; box-sizing: border-box; }
    body, html { width: 100%; height: 100%; background-color: var(--bg); color: var(--text); font-family: var(--font-body); overflow: hidden; user-select: none; }
    button { font-family: inherit; cursor: pointer; border: none; outline: none; background: none; }
    input { font-family: inherit; }
    .gradient-text { background: var(--gradient-text); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    .glass-card { background: var(--surface); backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px); border: 1px solid var(--border); border-radius: 20px; padding: 24px; box-shadow: 0 8px 32px rgba(0, 0, 0, 0.2); }
    .btn-gradient { background: var(--gradient-main); color: white; border-radius: 12px; padding: 14px 24px; font-weight: 700; transition: transform 0.15s, box-shadow 0.15s; display: flex; align-items: center; justify-content: center; gap: 12px; }
    .btn-gradient:active { transform: scale(0.97); }
    .btn-gradient:disabled { opacity: 0.4; pointer-events: none; }
    .btn-outline { border: 1px solid var(--border); color: var(--text); border-radius: 12px; padding: 10px 16px; font-weight: 600; transition: background 0.15s; }
    .btn-outline:active { transform: scale(0.97); }
    #app { width: 100vw; height: 100vh; position: relative; }
    .screen { position: absolute; top: 0; left: 0; right: 0; bottom: 0; display: flex; flex-direction: column; pointer-events: none; opacity: 0; transition: opacity 0.4s ease, transform 0.4s var(--spring); transform: translateY(20px); z-index: 10; }
    .screen.active { opacity: 1; pointer-events: auto; transform: translateY(0); }

    #home { padding: 20px; background-image: radial-gradient(circle at 50% 0%, rgba(110, 198, 245, 0.05), transparent 40%); }
    .top-bar { display: flex; justify-content: space-between; align-items: flex-start; margin-bottom: auto; z-index: 20; position: relative; }
    .top-center { display: flex; flex-direction: column; align-items: center; }
    .top-center h1 { font-size: 1.1rem; font-weight: 700; }
    .center-stage { margin: auto; display: flex; flex-direction: column; align-items: center; }
    .mic-wrapper { position: relative; width: 220px; height: 220px; display: flex; align-items: center; justify-content: center; }
    .wave-canvas { position: absolute; inset: 0; width: 100%; height: 100%; z-index: 1; pointer-events:none; }
    .mic-btn { width: 120px; height: 120px; border-radius: 50%; background: var(--bg); position: relative; z-index: 2; display: flex; align-items: center; justify-content: center; transition: transform 0.2s var(--spring); }
    .mic-btn::before { content: ''; position: absolute; inset: -4px; border-radius: 50%; background: var(--gradient-main); z-index: -1; }
    .mic-btn.recording { transform: scale(0.95); }
    .mic-btn.recording::before { background: linear-gradient(135deg, #ff4e50, #f9d423); animation: pulse-ring 1s infinite alternate; }
    .mic-icon { width: 40px; height: 40px; fill: var(--text); }
    .status-text { font-family: var(--font-mono); margin-top: 24px; font-size: 0.9rem; color: var(--accent3); height: 20px; }
    .bottom-actions { margin-top: auto; display: flex; flex-direction: column; align-items: center; gap: 16px; min-height: 80px; z-index: 20; }

    #analysis { background: rgba(8, 12, 24, 0.95); padding: 20px; display: flex; flex-direction: column; align-items: center; transform: translateY(100%); z-index: 200; }
    #analysis.active { transform: translateY(0); }
    .result-container { width: 100%; max-width: 480px; display: none; flex-direction: column; gap: 16px; margin: auto; padding-top: 60px; overflow-y: auto; }
    .result-container.show { display: flex; }
    .class-title { font-size: 2rem; font-weight: 800; margin-bottom: 4px; text-align: center; }
    .box-card { background: rgba(0,0,0,0.3); border: 1px solid rgba(255,255,255,0.05); padding: 16px; border-radius: 12px; margin-bottom: 4px; }
    .box-title { font-size: 0.75rem; text-transform: uppercase; color: var(--text-muted); font-family: var(--font-mono); }
    .alert-banner { display: none; background: rgba(231, 76, 60, 0.2); border: 1px solid var(--critical); color: #ff8a80; padding: 12px; border-radius: 12px; text-align: center; font-weight: 800; letter-spacing: 2px; }

    /* Toasts */
    #toast-container { position: fixed; top: 20px; right: 20px; z-index: 1000; display: flex; flex-direction: column; gap: 10px; }
    .toast { background: rgba(0,0,0,0.8); border-left: 4px solid var(--accent3); padding: 12px 20px; border-radius: 8px; color: white; display: flex; gap: 12px; box-shadow: 0 4px 12px rgba(0,0,0,0.3); }
    @keyframes pulse-ring { 0% { box-shadow: 0 0 0 0 rgba(255, 78, 80, 0.4); } 100% { box-shadow: 0 0 0 15px rgba(255, 78, 80, 0); } }
  </style>
</head>
<body>
  <div id="toast-container"></div>
  <div id="app">
    <!-- SCREEN Home -->
    <div id="home" class="screen active">
      <div class="top-bar"><div class="top-center"><h1 class="gradient-text">Sound classifier Native</h1></div></div>
      <div class="center-stage">
        <div class="mic-wrapper">
          <canvas class="wave-canvas" id="main-wave"></canvas>
          <button class="mic-btn" id="btn-record">
            <svg class="mic-icon" viewBox="0 0 24 24"><path d="M12 14c1.66 0 2.99-1.34 2.99-3L15 5c0-1.66-1.34-3-3-3S9 3.34 9 5v6c0 1.66 1.34 3 3 3zm5.3-3c0 3-2.54 5.1-5.3 5.1S6.7 14 6.7 11H5c0 3.41 2.72 6.23 6 6.72V21h2v-3.28c3.28-.48 6-3.3 6-6.72h-1.7z"/></svg>
          </button>
        </div>
        <div class="status-text" id="record-status-txt">Tap to Record (Flask connected)</div>
      </div>
      <div class="bottom-actions">
        <!-- Audio input hidden -->
        <input type="file" id="audio-upload" accept="audio/*" style="display:none;">
        <button class="btn-outline" id="btn-upload-file">Upload Audio File (.wav/.webm)</button>
        <button class="btn-gradient" id="btn-analyze" style="display:none; background: linear-gradient(135deg, #11998e, #38ef7d);">Confirm & Analyze</button>
      </div>
    </div>

    <!-- SCREEN 5: Analysis -->
    <div id="analysis" class="screen">
      <div class="result-container" id="analysis-result">
        <div class="alert-banner" id="res-alert-banner">⚠ THREAT DETECTED</div>
        <div class="class-title gradient-text" id="res-class">Standard Urban</div>

        <div class="box-card">
          <div class="box-title">Random Forest Confidence</div>
          <div style="font-family: var(--font-mono); font-size: 0.8rem;" id="res-conf-val">85%</div>
        </div>

        <div class="box-card">
          <div class="box-title">Description</div>
          <div class="box-content" id="res-desc">...</div>
        </div>

        <div style="display:flex; gap: 12px; margin-top: 10px;">
          <button class="btn-outline" id="btn-scan-again" style="flex:1;">Scan Again</button>
        </div>
      </div>
    </div>
  </div>

<script>
    const $ = id => document.getElementById(id);
    let currentAudio = null;
    let mediaRecorder;
    let recordedChunks = [];
    let isRecording = false;

    // File Upload Handler
    $('btn-upload-file').onclick = () => $('audio-upload').click();
    $('audio-upload').onchange = (e) => {
        const f = e.target.files[0];
        if(f) {
            currentAudio = { blob: f, name: f.name };
            $('record-status-txt').innerText = `Ready: ${f.name}`;
            $('btn-analyze').style.display = 'flex';
        }
    };

    // Mic handling (simplified for pure recording)
    async function startRecording() {
        try {
            const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
            mediaRecorder = new MediaRecorder(stream);
            recordedChunks = [];
            mediaRecorder.ondataavailable = e => { if (e.data.size>0) recordedChunks.push(e.data); };
            mediaRecorder.onstop = () => {
                const blob = new Blob(recordedChunks, { type: 'audio/webm' });
                currentAudio = { blob: blob, name: 'Recording.webm' };
                $('record-status-txt').innerText = `Recording captured`;
                $('btn-analyze').style.display = 'flex';
                stream.getTracks().forEach(t => t.stop());
            };
            mediaRecorder.start();
            isRecording = true;
            $('btn-record').classList.add('recording');
            $('record-status-txt').innerText = "Recording...";
        } catch(e) { }
    }

    $('btn-record').onmousedown = () => { if(!isRecording) startRecording(); };
    $('btn-record').onmouseup = () => { if(isRecording) { mediaRecorder.stop(); isRecording=false; $('btn-record').classList.remove('recording'); }};

    // Pushing data to Flask API
    $('btn-analyze').onclick = async () => {
        if(!currentAudio) return;
        $('analysis').classList.add('active');
        $('analysis-result').classList.remove('show');

        // Formulate FormData to send audio to python!
        const formData = new FormData();
        formData.append('audio', currentAudio.blob, currentAudio.name);

        try {
            const response = await fetch('/api/predict', { method: 'POST', body: formData });
            const data = await response.json();

            if(data.error) throw new Error(data.error);

            $('analysis-result').classList.add('show');
            $('res-class').innerText = data.prediction;
            $('res-conf-val').innerText = '95%'; // Mocked RF conf for now
            $('res-desc').innerText = `Your Machine Learning model classified this pattern natively.`;

            if(data.alert_level === 'CRITICAL') { $('res-alert-banner').style.display = 'block'; }
            else { $('res-alert-banner').style.display = 'none'; }

        } catch(e) {
            alert("Error in Model prediction: " + e.message);
        }
    };

    $('btn-scan-again').onclick = () => {
        $('analysis').classList.remove('active');
        $('btn-analyze').style.display = 'none';
        $('record-status-txt').innerText = 'Tap to Record (Flask connected)';
        currentAudio = null;
    };
</script>
</body>
</html>
"""

@app.route('/')
def home():
    return render_template_string(HTML_CONTENT)

@app.route('/api/predict', methods=['POST'])
def predict():
    if 'audio' not in request.files:
        return jsonify({"error": "No audio file provided"}), 400

    file = request.files['audio']
    temp_path = "temp_audio_upload.webm"
    file.save(temp_path)

    prediction = "Unrecognized"
    alert_lvl = "SAFE"
    try:
        # Load audio using librosa (uses default high-quality resampler instead of resampy)
        audio, sr = librosa.load(temp_path)
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        features = np.mean(mfccs, axis=1).reshape(1, -1)

        features_scaled = scaler.transform(features)
        prediction = model.predict(features_scaled)[0]

        # Simple threat mapping logic matching urban sound classes
        if prediction in ['siren', 'gun_shot']:
            alert_lvl = 'CRITICAL'
        elif prediction in ['drilling', 'jackhammer']:
            alert_lvl = 'WARNING'

    except Exception as e:
        return jsonify({"error": str(e)}), 500

    return jsonify({
        "prediction": prediction,
        "alert_level": alert_lvl
    })

# ==============================================================================
# RUNNING THE SERVER IN COLAB
# ==============================================================================

import logging
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

print("\n" + "="*80)
print("✅ TRAINING & SETUP COMPLETE!")

try:
    # This is the FREE, native way to get a public URL inside Google Colab!
    from google.colab.output import eval_js
    public_url = eval_js("google.colab.kernel.proxyPort(5000)")
    print(f"🚀 YOUR WEB APP IS NOW LIVE AT THIS PUBLIC LINK:")
    print(f">> CLICK HERE: {public_url} <<")
except Exception as e:
    print(f">> IF YOU ARE NOT IN COLAB, CLICK HERE: http://127.0.0.1:5000 <<")

print("="*80)
print("Click the public link above to open your Audio classification App!\n")

# Start the Flask web server
app.run(port=5000)

Loading model for Web Service...

✅ TRAINING & SETUP COMPLETE!
🚀 YOUR WEB APP IS NOW LIVE AT THIS PUBLIC LINK:
>> CLICK HERE: https://5000-m-s-kkb-usw4c2-1zilgu0kwyagh-c.us-west4-2.prod.colab.dev <<
Click the public link above to open your Audio classification App!

 * Serving Flask app '__main__'
 * Debug mode: off


/tmp/ipykernel_1732/2385932340.py:295: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(temp_path)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1732/2385932340.py:295: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(temp_path)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1732/2385932340.py:295: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(temp_path)
/usr/local/lib/python3.12/dist-packages/librosa/core/a